# Commit 08: dataset modelowy ML/QNN

Notebook przygotowuje i dokumentuje pierwszy dataset `company-year` gotowy do dalszych eksperymentow ML/QNN w pracy magisterskiej SGH. Etap obejmuje budowe targetu `t+1`, podzial train/validation/test, kontrole jakosci, EDA oraz wykresy diagnostyczne.

Ten notebook nie trenuje modeli, nie wykonuje finalnej imputacji, nie skaluje feature'ow i nie przygotowuje jeszcze obwodow QNN.

## 1. Uruchomienie deterministycznego buildera

Logika produkcji CSV i raportow jest wydzielona do `src/data/modeling_dataset.py`, a `src/data/08_build_modeling_dataset.py` pozostaje cienkim runnerem pipeline'u. Notebook pozostaje glownym miejscem eksploracji, kontroli i dokumentacji przed-eksperymentalnej.

In [ ]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd()
if not (BASE_DIR / "src").exists():
    BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.data.modeling_dataset import build_modeling_dataset

build_result = build_modeling_dataset()
build_result

## 2. Wczytanie artefaktow Commit 08

In [ ]:
import pandas as pd
from IPython.display import display

modeling_path = BASE_DIR / "data" / "processed" / "modeling_dataset.csv"
excluded_path = BASE_DIR / "data" / "processed" / "modeling_dataset_excluded.csv"
split_summary_path = BASE_DIR / "data" / "reports" / "modeling_dataset_split_summary.csv"
feature_coverage_path = BASE_DIR / "data" / "reports" / "modeling_dataset_feature_coverage.csv"
warning_policy_path = BASE_DIR / "data" / "reports" / "modeling_dataset_warning_policy.csv"
quality_report_path = BASE_DIR / "data" / "reports" / "modeling_dataset_quality_report.md"
source_wide_path = BASE_DIR / "data" / "interim" / "sec_facts_wide.csv"
warnings_path = BASE_DIR / "data" / "reports" / "sec_facts_sanity_warnings.csv"

modeling = pd.read_csv(modeling_path)
excluded = pd.read_csv(excluded_path)
split_summary = pd.read_csv(split_summary_path)
feature_coverage = pd.read_csv(feature_coverage_path)
warning_policy = pd.read_csv(warning_policy_path)
source_wide = pd.read_csv(source_wide_path)
warnings = pd.read_csv(warnings_path)

print(f"modeling_dataset: {modeling.shape}")
print(f"modeling_dataset_excluded: {excluded.shape}")
print(f"feature years: {modeling['company_year'].min()}-{modeling['company_year'].max()}")
print(f"companies: {modeling['cik10'].nunique():,}")

## 3. Splity i target

In [ ]:
display(split_summary)
pd.crosstab(
    modeling["split"],
    modeling["financial_deterioration_next_year"],
    margins=True,
)

## 4. Porownanie splitow wedlug sektorow i lat

In [ ]:
sector_summary = (
    modeling.groupby(["split", "research_sector"])
    .agg(
        row_count=("cik10", "size"),
        company_count=("cik10", "nunique"),
        positive_target_ratio=("financial_deterioration_next_year", "mean"),
    )
    .reset_index()
    .sort_values(["split", "row_count"], ascending=[True, False])
)
display(sector_summary)

company_years_by_year = modeling.groupby(["company_year", "split"]).size().unstack(fill_value=0)
display(company_years_by_year)

## 5. Coverage feature'ow po czyszczeniu

In [ ]:
feature_coverage_all = (
    feature_coverage[feature_coverage["split"].eq("all")]
    .sort_values("coverage_ratio")
    .reset_index(drop=True)
)
display(feature_coverage_all)

feature_columns = feature_coverage_all["feature"].tolist()
modeling[feature_columns].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

## 6. Braki danych i rozklady zmiennych finansowych

In [ ]:
missing_by_year = (
    modeling.groupby("company_year")[feature_columns]
    .apply(lambda frame: frame.isna().mean())
    .reset_index()
)
display(missing_by_year)

financial_columns = ["assets", "liabilities", "revenues", "net_income", "equity", "cash", "operating_costs"]
modeling[financial_columns].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

## 7. Warningi z Commit 07 i powody wykluczen

In [ ]:
display(warning_policy.sort_values(["policy", "warning_count"], ascending=[True, False]))

exclusion_counts = (
    excluded["exclusion_reasons"]
    .dropna()
    .str.split(";")
    .explode()
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="count")
)
display(exclusion_counts)

## 8. Kontrole leakage i zgodnosci z konfiguracja

In [ ]:
assert modeling["company_year"].between(2011, 2024).all()
assert not modeling["company_year"].eq(2025).any()
assert set(modeling.loc[modeling["company_year"].eq(2024), "target_year"].dropna().unique()) <= {2025}
assert set(modeling.loc[modeling["split"].eq("train"), "company_year"].dropna().unique()) <= set(range(2011, 2021))
assert set(modeling.loc[modeling["split"].eq("validation"), "company_year"].dropna().unique()) <= {2021, 2022}
assert set(modeling.loc[modeling["split"].eq("test"), "company_year"].dropna().unique()) <= {2023, 2024}
assert modeling["missing_feature_ratio"].max() <= 0.2

checks = {
    "max_feature_year": int(modeling["company_year"].max()),
    "rows_2025_in_modeling_dataset": int(modeling["company_year"].eq(2025).sum()),
    "rows_2024_in_test": int(modeling["company_year"].eq(2024).sum()),
    "target_years_for_2024": sorted(modeling.loc[modeling["company_year"].eq(2024), "target_year"].dropna().unique().tolist()),
    "max_missing_feature_ratio": float(modeling["missing_feature_ratio"].max()),
}
checks

## 9. Korelacje feature'ow

In [ ]:
modeling[feature_columns].corr(numeric_only=True).round(3)

## 10. Wykresy diagnostyczne

In [ ]:
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt

figure_dir = BASE_DIR / "reports" / "figures" / "commit_8"
figure_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def save_figure(name):
    path = figure_dir / name
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    plt.close()
    return path

feature_year_source = source_wide[source_wide["company_year"].between(2011, 2024)]
year_counts = pd.DataFrame({
    "source_feature_years": feature_year_source["company_year"].value_counts().sort_index(),
    "modeling_dataset": modeling["company_year"].value_counts().sort_index(),
}).fillna(0).astype(int)
year_counts.plot(kind="bar")
plt.title("Company-years by feature year")
plt.xlabel("feature year")
plt.ylabel("company-years")
save_figure("company_years_by_year.png")

(feature_coverage_all.set_index("feature")["coverage_ratio"].sort_values() * 100).plot(kind="barh")
plt.title("Feature coverage after cleaning")
plt.xlabel("coverage (%)")
plt.ylabel("")
save_figure("feature_coverage_after_cleaning.png")

warnings["check_name"].value_counts().head(20).sort_values().plot(kind="barh")
plt.title("Warnings by check_name")
plt.xlabel("warning count")
plt.ylabel("")
save_figure("warnings_by_check_name.png")

pd.crosstab(modeling["split"], modeling["financial_deterioration_next_year"]).plot(kind="bar")
plt.title("Target distribution by split")
plt.xlabel("split")
plt.ylabel("observations")
plt.legend(title="target")
save_figure("target_distribution_by_split.png")

ratio_columns = ["debt_to_assets", "roa", "profit_margin", "current_ratio"]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for axis, column in zip(axes.ravel(), ratio_columns):
    values = pd.to_numeric(modeling[column], errors="coerce").replace([float("inf"), float("-inf")], pd.NA).dropna()
    lower, upper = values.quantile([0.01, 0.99])
    axis.hist(values.clip(lower=lower, upper=upper), bins=40)
    axis.set_title(column)
    axis.set_xlabel("value clipped to p01-p99")
    axis.set_ylabel("count")
save_figure("ratio_feature_distributions.png")

(modeling[feature_columns].isna().mean().sort_values() * 100).plot(kind="barh")
plt.title("Missing values by feature")
plt.xlabel("missing (%)")
plt.ylabel("")
save_figure("missing_values_by_feature.png")

exclusion_counts.set_index("reason")["count"].sort_values().plot(kind="barh")
plt.title("Exclusions by reason")
plt.xlabel("excluded observations")
plt.ylabel("")
save_figure("exclusions_by_reason.png")

corr = modeling[feature_columns].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(feature_columns)))
ax.set_xticklabels(feature_columns, rotation=45, ha="right")
ax.set_yticks(range(len(feature_columns)))
ax.set_yticklabels(feature_columns)
ax.set_title("Feature correlation heatmap")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
save_figure("correlation_heatmap.png")

for figure_path in sorted(figure_dir.glob("*.png")):
    display(Markdown(f"### {figure_path.name}"))
    display(Image(filename=str(figure_path)))

## 11. Decyzje metodologiczne do zatwierdzenia przez autora

- Minimalne mianowniki ratio: 1 000 USD dla `assets`, `revenues`, `current_liabilities` i dodatniego `equity`.
- `hard_exclude_company_year` usuwa obserwacje z istotnymi problemami technicznymi lub bilansowymi.
- `feature_level_cleanup` ustawia tylko wskazana zmienna na missing, bez automatycznego dropu calego company-year.
- Spolki z niskim coverage dostaja flagi diagnostyczne, a nie automatyczny drop calej spolki.
- Dataset nie wykonuje finalnej imputacji ani winsoryzacji; te decyzje powinny byc podjete przed pierwszym eksperymentem modelowym.
- Target `financial_deterioration_next_year` wymaga co najmniej dwoch poprawnych warunkow pogorszenia z konfiguracji.
- Rok 2025 sluzy tylko do targetu dla obserwacji 2024 i nie jest rokiem feature'ow.

## 12. Raport jakosci

Pelny raport techniczny znajduje sie w `data/reports/modeling_dataset_quality_report.md`. Raport powinien byc traktowany jako dokumentacja procesu przygotowania danych, a nie jako gotowa interpretacja wynikow badawczych.